# SPATIAL INTELLIGENCE — MULTI-FLOOR BUILDING (Marsella / Unité d'Habitation)

Analyze **three stacked floor plans** of Le Corbusier's *Unité d'Habitation* (Marseille) as a single connected building and run the Spatial Intelligence workflows (Session 03 / Assignment 02) **across all floors simultaneously**.

**Geometry note.** The source `Marsella_3-Floor-Plans.obj` contains three 2D plans. After importing with `Topology.ByOBJPath` (the project's Z-up convention) the three plans lie flat in the **XY plane** at three discrete levels **Z = 0, 4, 8**. Each plan is a long slab; its walls and stair shafts are the *holes* (the *Therme Vals* approach suggested by the instructor).

**Method (instructor's grid-sampling strategy).** For every floor we overlay a regular grid and keep the grid points that fall inside the navigable (meshed) area. Those points become graph nodes; neighbouring valid points are joined by edges. The three floor graphs are then **stacked** and **stitched together through stair nodes**, so that connectivity metrics such as **Degree Centrality are computed over the WHOLE building, not floor-by-floor.**

Workflows included: Shortest Path (cross-floor), Closeness Centrality (Integration), Betweenness Centrality (Choice), Community Detection and **building-wide Degree Centrality**.

## 1. Import the needed libraries

In [1]:
import os, math, time
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

e:\IAAC Local GIT Repositories\Graph ML - Environment\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy version

In [2]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.43) is EQUAL TO the latest version available on PyPI.


## 3. Configuration

In [3]:
renderer = "vscode"   # use "notebook" or "browser" if figures do not appear

# Paths
BASE_DIR  = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Final_Project"
OBJ_PATH  = os.path.join(BASE_DIR, "3D-Models", "Marsella_3-Floor-Plans.obj")
ASSETS_DIR = os.path.join(BASE_DIR, "assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

# Floor levels (Z value of each plan after import) ordered bottom -> top
FLOOR_LEVELS = [0, 4, 8]
FLOOR_NAMES  = ["Floor 1", "Floor 2", "Floor 3"]

# Analysis grid spacing (plan units). Smaller = finer & slower. 3.0 is a good balance.
GRID_SIZE = 3.0

# Vertical spacing used to STACK the floors in the combined 3D graph (visual only)
FLOOR_HEIGHT = 15.0

# Stair / vertical-circulation locations in PLAN coordinates (u = X, v = Y after import).
# These are where the floors get connected to each other. Edit them to match your stairs.
# (Run section 7 first to see the navigable grid with coordinates, then refine these.)
STAIR_LOCATIONS = [
    (155.0, 345.0),
    (205.0, 345.0),
    (245.0, 345.0),
]

SAVE_IMAGES = True

def save_fig(fig, filename):
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1800, height=900, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

* `extract_triangles` — pull the triangulated faces of one floor into a NumPy array of plan-space triangles.
* `points_inside` — vectorised point-in-mesh test (keeps only navigable grid points).
* `find_closest_node` — the instructor's `find_closest_vertex`, adapted to plan coordinates: snaps a stair location to the nearest grid node of a floor.
* `color_graph_vertices` / `plot_floor_heatmaps` — visual helpers.

In [4]:
def extract_triangles(face_list):
    # Return (T,3,2) array of plan-space (X,Y) triangles for a list of topologic faces.
    tris = []
    for f in face_list:
        vs = Topology.Vertices(f)
        pts = [(Vertex.X(v), Vertex.Y(v)) for v in vs]
        for i in range(1, len(pts) - 1):          # fan-triangulate (faces are already triangles)
            tris.append([pts[0], pts[i], pts[i + 1]])
    return np.array(tris)

def points_inside(tris, P):
    # Boolean mask: which points in P (N,2) fall inside ANY triangle of tris (T,3,2).
    a, b, c = tris[:, 0], tris[:, 1], tris[:, 2]
    v0 = b - a; v1 = c - a
    d00 = (v0 * v0).sum(1); d01 = (v0 * v1).sum(1); d11 = (v1 * v1).sum(1)
    den = d00 * d11 - d01 * d01
    den[den == 0] = 1e-12
    inside = np.zeros(len(P), bool)
    for i, p in enumerate(P):
        v2 = p - a
        d20 = (v2 * v0).sum(1); d21 = (v2 * v1).sum(1)
        u = (d11 * d20 - d01 * d21) / den
        w = (d00 * d21 - d01 * d20) / den
        if np.any((u >= -1e-6) & (w >= -1e-6) & (u + w <= 1 + 1e-6)):
            inside[i] = True
    return inside

def find_closest_node(node_xy, x, y):
    # Index of the grid node closest to (x, y) -- the instructor's find_closest_vertex.
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def color_graph_vertices(gverts, values, key, colorScale="viridis"):
    # Store a per-vertex colour (and raw value) on the graph vertices for Topology.Show.
    mn, mx = float(min(values)), float(max(values))
    if mx == mn: mx = mn + 1e-9
    for v, val in zip(gverts, values):
        d = Topology.Dictionary(v)
        col = Color.AnyToHex(Color.ByValueInRange(float(val), minValue=mn, maxValue=mx, colorScale=colorScale))
        d = Dictionary.SetValueAtKey(d, key, col)
        d = Dictionary.SetValueAtKey(d, key + "_val", float(val))
        v = Topology.SetDictionary(v, d)

def plot_floor_heatmaps(gverts, values, title, filename, colorscale="Viridis"):
    # One 2D scatter per floor, coloured by a building-wide metric.
    xs = np.array([Vertex.X(v) for v in gverts])
    ys = np.array([Vertex.Y(v) for v in gverts])
    fl = np.array([int(round(Vertex.Z(v) / FLOOR_HEIGHT)) for v in gverts])
    vals = np.array(values, dtype=float)
    n = len(FLOOR_LEVELS)
    fig = make_subplots(rows=n, cols=1, subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(n)])
    for i in range(n):
        m = fl == i
        fig.add_trace(go.Scatter(x=xs[m], y=ys[m], mode="markers",
                                 marker=dict(size=7, color=vals[m], colorscale=colorscale,
                                             showscale=(i == 0), colorbar=dict(title="value", len=0.9)),
                                 showlegend=False), row=i + 1, col=1)
        fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
    fig.update_layout(title=title, height=320 * n, width=1500, plot_bgcolor="white")
    fig.show(renderer=renderer)
    save_fig(fig, filename)
    return fig

## 5. Import the OBJ and split it into the three floor plans

`Topology.ByOBJPath` returns a list of clusters of triangulated faces. We collect every face and bin it by its centroid's Z value into the three floor levels.

In [ ]:
result = Topology.ByOBJPath(OBJ_PATH)
all_faces = []
for item in result:
    if Topology.IsInstance(item, "Cluster"):
        cf = Cluster.Faces(item)
        if cf: all_faces.extend(cf)
    elif Topology.IsInstance(item, "Face"):
        all_faces.append(item)
print(f"Imported {len(all_faces)} triangulated faces")

floor_faces = {lv: [] for lv in FLOOR_LEVELS}
for f in all_faces:
    z = Vertex.Z(Topology.Centroid(f))
    lv = min(FLOOR_LEVELS, key=lambda k: abs(k - z))
    floor_faces[lv].append(f)
for i, lv in enumerate(FLOOR_LEVELS):
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_faces[lv])} faces")

## 6. Show the three raw floor plans

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    for t in tris:
        xs = list(t[:, 0]) + [t[0, 0]]
        ys = list(t[:, 1]) + [t[0, 1]]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", fill="toself",
                                 line=dict(color="rgba(0,0,0,0.35)", width=0.4),
                                 fillcolor="rgba(60,110,230,0.25)", showlegend=False), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_layout(title="Three imported floor plans (plan view)", height=300 * len(FLOOR_LEVELS),
                  width=1500, plot_bgcolor="white")
fig.show(renderer=renderer)
save_fig(fig, "01_floor_plans.png")

## 7. Sample a navigable grid on each floor

For every floor we lay a regular grid over its bounding box and keep only the points that fall inside the meshed (navigable) area. These valid points become the graph nodes of that floor.

In [ ]:
# Common plan bounding box (shared by all floors so the grids line up vertically)
allxy = np.vstack([extract_triangles(floor_faces[lv]).reshape(-1, 2) for lv in FLOOR_LEVELS])
UMIN, VMIN = allxy.min(0)
UMAX, VMAX = allxy.max(0)
us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])
print(f"Plan box: u[{UMIN:.1f},{UMAX:.1f}]  v[{VMIN:.1f},{VMAX:.1f}]  ->  {len(GRID_PTS)} candidate points/floor")

floor_valid = {}    # level -> (valid_xy ndarray, index_map dict)
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    mask = points_inside(tris, GRID_PTS)
    valid = GRID_PTS[mask]
    floor_valid[lv] = valid
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(valid)} navigable nodes")

## 8. Show the navigable grids (use these coordinates to locate your stairs)

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    fig.add_trace(go.Scatter(x=valid[:, 0], y=valid[:, 1], mode="markers",
                             marker=dict(size=5, color="royalblue"), showlegend=False), row=i + 1, col=1)
    sx = [s[0] for s in STAIR_LOCATIONS]; sy = [s[1] for s in STAIR_LOCATIONS]
    fig.add_trace(go.Scatter(x=sx, y=sy, mode="markers",
                             marker=dict(size=14, color="red", symbol="x"), name="stairs",
                             showlegend=(i == 0)), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_layout(title="Navigable grids + stair locations (red x)", height=300 * len(FLOOR_LEVELS),
                  width=1500, plot_bgcolor="white")
fig.show(renderer=renderer)
save_fig(fig, "02_navigable_grids.png")

## 9. Build the per-floor graphs and stack them

Each floor's valid points are turned into topologic vertices placed at `Z = floor_index * FLOOR_HEIGHT`. Horizontal edges join 4-neighbour valid points. We accumulate everything into `all_v` / `all_e`.

In [ ]:
all_v = []          # topologic vertices (all floors)
all_e = []          # topologic edges
floor_index_map = {}  # level -> dict {(round u, round v): global vertex index}

def rk(u, v): return (round(float(u), 3), round(float(v), 3))

for fi, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    z = fi * FLOOR_HEIGHT
    idx = {}
    for (u, v) in valid:
        idx[rk(u, v)] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z)))
    floor_index_map[lv] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f"  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges")
print(f"Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges")

## 10. Connect the floors through the stairs

For every stair location and every pair of adjacent floors we snap to the closest navigable node on each floor (`find_closest_node`) and add a **vertical stair edge**. This is what turns three separate plans into one connected building.

In [ ]:
stair_node_pairs = []
for (sx, sy) in STAIR_LOCATIONS:
    for a, b in zip(FLOOR_LEVELS[:-1], FLOOR_LEVELS[1:]):
        va, vb = floor_valid[a], floor_valid[b]
        ia = find_closest_node(va, sx, sy)
        ib = find_closest_node(vb, sx, sy)
        gia = floor_index_map[a][rk(va[ia, 0], va[ia, 1])]
        gib = floor_index_map[b][rk(vb[ib, 0], vb[ib, 1])]
        all_e.append(Edge.ByVertices([all_v[gia], all_v[gib]]))
        stair_node_pairs.append((gia, gib))
print(f"Added {len(stair_node_pairs)} vertical stair edges "
      f"({len(STAIR_LOCATIONS)} stairs x {len(FLOOR_LEVELS)-1} floor gaps)")

## 11. Build the combined BUILDING graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f"Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)")
print(f"Graph density:  {Graph.Density(building_graph):.5f}")

## 12. Show the combined 3D building graph

Three stacked floors connected by the vertical stair edges (highlighted).

In [ ]:
fig = Topology.Show(building_graph,
                   vertexSize=3, vertexColor="royalblue",
                   edgeColor="lightgrey", edgeWidth=1,
                   backgroundColor="white",
                   width=1400, height=900,
                   showFigure=False, renderer=renderer)
# overlay stair edges in red
for (a, b) in stair_node_pairs:
    pa, pb = all_v[a], all_v[b]
    fig.add_trace(go.Scatter3d(x=[Vertex.X(pa), Vertex.X(pb)], y=[Vertex.Y(pa), Vertex.Y(pb)],
                               z=[Vertex.Z(pa), Vertex.Z(pb)], mode="lines",
                               line=dict(color="red", width=6), showlegend=False))
fig.show(renderer=renderer)
save_fig(fig, "03_building_graph_3d.png")

## 13. Building-wide DEGREE CENTRALITY

> **This is the key building-scale metric.** Degree centrality is computed on the **whole-building graph** (all three floors connected through the stairs), so stair nodes and well-connected circulation spaces score across floors — not per plan in isolation.

In [ ]:
t0 = time.time()
degree_values = Graph.DegreeCentrality(building_graph, normalize=True)
print(f"Degree centrality computed for {len(degree_values)} nodes "
      f"(range {min(degree_values):.4f} - {max(degree_values):.4f}, {time.time()-t0:.1f}s)")

color_graph_vertices(gverts, degree_values, "dc_color", colorScale="viridis")
plot_floor_heatmaps(gverts, degree_values, "Degree Centrality (whole building)",
                    "04_degree_centrality.png", colorscale="Viridis")

### Degree centrality on the 3D building

In [ ]:
fig = Topology.Show(building_graph,
                   vertexSize=4, vertexColorKey="dc_color",
                   edgeColor="lightgrey", edgeWidth=1,
                   backgroundColor="white", width=1400, height=900,
                   showFigure=False, renderer=renderer)
fig.show(renderer=renderer)
save_fig(fig, "05_degree_centrality_3d.png")

## 14. Closeness Centrality (Integration)

How close each space is to every other space in the **entire building**. High values = globally integrated, easy-to-reach locations.

In [ ]:
t0 = time.time()
closeness_values = Graph.ClosenessCentrality(building_graph)
print(f"Closeness centrality: {len(closeness_values)} nodes "
      f"(range {min(closeness_values):.4f} - {max(closeness_values):.4f}, {time.time()-t0:.1f}s)")
color_graph_vertices(gverts, closeness_values, "cc_color", colorScale="thermal")
plot_floor_heatmaps(gverts, closeness_values, "Closeness Centrality / Integration (whole building)",
                    "06_closeness_centrality.png", colorscale="Turbo")

## 15. Betweenness Centrality (Choice)

How often each space lies on the shortest paths between all other spaces. High values = critical circulation routes; the stair nodes typically light up because every cross-floor trip passes through them.

In [ ]:
t0 = time.time()
betweenness_values = Graph.BetweennessCentrality(building_graph, normalize=True)
print(f"Betweenness centrality: {len(betweenness_values)} nodes "
      f"(range {min(betweenness_values):.4f} - {max(betweenness_values):.4f}, {time.time()-t0:.1f}s)")
color_graph_vertices(gverts, betweenness_values, "bc_color", colorScale="thermal")
plot_floor_heatmaps(gverts, betweenness_values, "Betweenness Centrality / Choice (whole building)",
                    "07_betweenness_centrality.png", colorscale="Turbo")

## 16. Community Detection

Partition the building into spatial communities — densely connected groups of nodes. Because the graph spans all floors, a community can extend vertically through a stair.

In [ ]:
t0 = time.time()
community_values = Graph.CommunityPartition(building_graph)
n_comm = len(set(community_values))
print(f"Detected {n_comm} communities ({time.time()-t0:.1f}s)")
color_graph_vertices(gverts, community_values, "cp_color", colorScale="rainbow")
plot_floor_heatmaps(gverts, community_values, f"Community Detection — {n_comm} communities (whole building)",
                    "08_communities.png", colorscale="Rainbow")

### Communities on the 3D building

In [ ]:
fig = Topology.Show(building_graph,
                   vertexSize=4, vertexColorKey="cp_color",
                   edgeColor="lightgrey", edgeWidth=1,
                   backgroundColor="white", width=1400, height=900,
                   showFigure=False, renderer=renderer)
fig.show(renderer=renderer)
save_fig(fig, "09_communities_3d.png")

## 17. Cross-floor Shortest Path

Navigate from one end of the **bottom** floor to the far end of the **top** floor. The route is forced to climb through the stair nodes, demonstrating that the three plans are genuinely connected.

In [ ]:
def graph_closest(x, y, z):
    best, bd = None, 1e18
    for v in gverts:
        d = (Vertex.X(v) - x) ** 2 + (Vertex.Y(v) - y) ** 2 + (Vertex.Z(v) - z) ** 2
        if d < bd: bd, best = d, v
    return best

vmid = 0.5 * (VMIN + VMAX)
start_v = graph_closest(UMIN + GRID_SIZE, vmid, 0)
end_v   = graph_closest(UMAX - GRID_SIZE, vmid, (len(FLOOR_LEVELS) - 1) * FLOOR_HEIGHT)

t0 = time.time()
path = Graph.ShortestPath(building_graph, vertexA=start_v, vertexB=end_v)
if path is None:
    print("No path found — check that STAIR_LOCATIONS land on navigable nodes and that the floors are connected.")
else:
    print(f"Cross-floor shortest path: length {Wire.Length(path):.1f} ({time.time()-t0:.1f}s)")

### Show the cross-floor path on the building graph

In [ ]:
fig = Topology.Show(building_graph,
                   vertexSize=2, vertexColor="lightgrey",
                   edgeColor="rgba(200,200,200,0.5)", edgeWidth=1,
                   backgroundColor="white", width=1400, height=900,
                   showFigure=False, renderer=renderer)
if path is not None:
    pv = Topology.Vertices(path)
    fig.add_trace(go.Scatter3d(x=[Vertex.X(v) for v in pv], y=[Vertex.Y(v) for v in pv],
                               z=[Vertex.Z(v) for v in pv], mode="lines+markers",
                               line=dict(color="red", width=6), marker=dict(size=3, color="red"),
                               showlegend=False))
    for v, c in [(start_v, "green"), (end_v, "blue")]:
        fig.add_trace(go.Scatter3d(x=[Vertex.X(v)], y=[Vertex.Y(v)], z=[Vertex.Z(v)],
                                   mode="markers", marker=dict(size=8, color=c), showlegend=False))
fig.show(renderer=renderer)
save_fig(fig, "10_shortest_path_3d.png")

## 18. Visibility / Isovist Analysis (per floor)

Unlike the connectivity metrics above, **visibility does not cross floors** — you cannot see the floor above through the slab. So this Visibility Graph Analysis (VGA) is computed **independently for each floor**.

We scatter a coarse grid of *isovist viewpoints* over the navigable area and connect every pair of viewpoints whose straight sightline stays inside the floor (walls / voids block it). The **visibility degree** of a viewpoint = how many other viewpoints it can see. High values = long, open sightlines (corridors, the internal *rue*); low values = enclosed corners.

In [ ]:
# Isovist viewpoints are sampled on a COARSER grid than the analysis nodes.
VGA_GRID_SIZE = 8.0    # spacing of the isovist viewpoints
VIS_SAMPLES   = 12     # samples along each sightline for the occlusion test

def compute_visibility(tris, viewpoints):
    # Visibility degree of each viewpoint: how many other viewpoints have an
    # unobstructed sightline (sampled segment stays inside the navigable mesh).
    n = len(viewpoints)
    ts = np.linspace(0.12, 0.88, VIS_SAMPLES)
    deg = np.zeros(n, int)
    for i in range(n):
        for j in range(i + 1, n):
            seg = viewpoints[i][None, :] * (1 - ts)[:, None] + viewpoints[j][None, :] * ts[:, None]
            if points_inside(tris, seg).all():
                deg[i] += 1; deg[j] += 1
    return deg

uvp = np.arange(UMIN, UMAX + VGA_GRID_SIZE, VGA_GRID_SIZE)
vvp = np.arange(VMIN, VMAX + VGA_GRID_SIZE, VGA_GRID_SIZE)
UUv, VVv = np.meshgrid(uvp, vvp)
VP_PTS = np.column_stack([UUv.ravel(), VVv.ravel()])

visibility = {}   # level -> (viewpoints_xy, visibility_degree)
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    vp = VP_PTS[points_inside(tris, VP_PTS)]
    t0 = time.time()
    deg = compute_visibility(tris, vp)
    visibility[lv] = (vp, deg)
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(vp)} viewpoints, "
          f"visibility degree {deg.min()}-{deg.max()} (mean {deg.mean():.1f}, {time.time()-t0:.1f}s)")

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    vp, deg = visibility[lv]
    fig.add_trace(go.Scatter(x=vp[:, 0], y=vp[:, 1], mode="markers",
                             marker=dict(size=13, color=deg, colorscale="Plasma",
                                         showscale=(i == 0), colorbar=dict(title="visibility", len=0.9)),
                             showlegend=False), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_layout(title="Visibility Graph Analysis — isovist degree (per floor)",
                  height=320 * len(FLOOR_LEVELS), width=1500, plot_bgcolor="white")
fig.show(renderer=renderer)
save_fig(fig, "11_visibility_isovist.png")

## 19. Building-wide summary

A compact table of the building-scale metrics. All values are derived from the single connected building graph, so they describe the *whole* Unité d'Habitation section rather than any individual floor.

In [ ]:
def top_floor_breakdown(values):
    fl = np.array([int(round(Vertex.Z(v) / FLOOR_HEIGHT)) for v in gverts])
    vals = np.array(values, dtype=float)
    return [round(float(vals[fl == i].mean()), 4) for i in range(len(FLOOR_LEVELS))]

print(f"{'Metric':<24}{'min':>10}{'max':>10}{'mean':>10}   per-floor mean")
for name, vals in [("Degree centrality", degree_values),
                   ("Closeness centrality", closeness_values),
                   ("Betweenness centrality", betweenness_values)]:
    a = np.array(vals, dtype=float)
    print(f"{name:<24}{a.min():>10.4f}{a.max():>10.4f}{a.mean():>10.4f}   {top_floor_breakdown(vals)}")
print()
print(f"Nodes: {len(gverts)} | Edges: {len(gedges)} | Density: {Graph.Density(building_graph):.5f} "
      f"| Communities: {len(set(community_values))} | Stair edges: {len(stair_node_pairs)}")